**Mount Google Drive with the Google Colab Notebook**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/yolo26"

In [ ]:
! mkdir -p "plants_classification"

In [ ]:
%cd "/content/drive/MyDrive/yolo26/plants_classification"

**Install All the Required Packages**

In [ ]:
! pip install ultralytics

**Import All the Required Libraries**

In [ ]:
import ultralytics

In [ ]:
from ultralytics import YOLO

In [ ]:
from IPython.display import Image

**Check Ultralytics Version**

In [ ]:
ultralytics.checks()

**Download the Dataset from Roboflow**

In [ ]:
! pip install roboflow --only-binary :all:

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="xxxxxxxxxxxxxxxxxxxxx")
project = rf.workspace("rockpaperscissors-adtav").project("plants-classification-zcckg")
version = project.version(3)
dataset = version.download("folder")

**Train / Fine-Tune the YOLO26 Classification Model**

In [ ]:
# Start training from a pretrained *.pt model
! yolo task=classify mode=train data="plants-classification-3" model="yolo26n-cls.pt" epochs=40 imgsz=640

**Download the Model Weights from Drive**

In [ ]:
! gdown "https://drive.google.com/uc?id=1bzBLmOigmM1LJX3GbA7tuJL1mNHxBBZr&confirm=t"

**Results Analysis**

**Confusion Matrix**

In [ ]:
Image("runs/classify/train/confusion_matrix.png", width = 800)

**Normalized Confusion Matrix**

**Model Prediction on Validation Batch**

In [ ]:
Image("runs/classify/train/val_batch0_pred.jpg", width=800)

In [ ]:
Image("runs/classify/train/val_batch1_pred.jpg", width=800)

**Training and Validation Loss**

In [ ]:
Image("runs/classify/train/results.png", width = 800)

**Predict on Test Data using Trained Model**

In [ ]:
! pwd

In [ ]:
# Load the Trained Classification Model
model = YOLO("best.pt")

In [ ]:
! find plants-classification-3/test -type f | wc -l

In [ ]:
# Loop through each class folder in the test directory
import os

# Specify the test folder path
test_folder = "plants-classification-3/test"

for class_folder in os.listdir(test_folder):
  class_folder_path = os.path.join(test_folder, class_folder)
  # print(class_folder_path)
  # Make sure its a directory
  if os.path.isdir(class_folder_path):
    ground_truth_class = class_folder # This is the ground truth class based on the folder name
    print(f"Processing Class Folder (Ground Truth): {ground_truth_class}")
    # List All Images in the Current Class Folder
    image_files = [os.path.join(class_folder_path, img) for img in os.listdir(class_folder_path) if img.endswith(('.jpg', '.jpeg', '.png'))]
    # Predict on each image in the class folder
    for img_path in image_files:
      # Perform Classification on the Image
      results = model.predict(img_path, conf = 0.25)
      # Loop through the results and extract predictions
      for result in results:
        # Get the predicted class index (top-1) and confidence
        predicted_class_index = result.probs.top1 # Get the predicted class index (top-1)
        predicted_class = result.names[predicted_class_index] # Map index to class name
        confidence = result.probs.top1conf # Get the confidence score for the top prediction
        # Display the prediction results for the image along with the ground truths
        print(f"Image: {img_path}")
        print(f"Ground Truth: {ground_truth_class}, Predicted Class: {predicted_class}, Confidence: {confidence:.2f}")
